# Milestone 2 — LaTeX Parsing, Standardization and Deduplication

**CSC14119 — Introduction to Data Science · Group 3**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuyNhat1705/Intro2DS-Group3-Source/blob/main/src/parser.ipynb)

This notebook runs the pipeline described in `docs/requirement/parsing.md`: **gather files → clean up → build the tree and extract references → unify the bibliography → rename citation keys → deduplicate elements → export tables and statistics**.

The input is the existing Milestone 1 corpus; the notebook does not download arXiv again. Edit the configuration cell, then choose **Run all**. The scope follows the configuration cell; set `SAMPLE_N` to try a small sample before the full run. SaT runs on CPU and downloads its weights at the first sentence split if they are not cached yet.

## Method overview

This section summarizes how each requirement of the milestone is implemented; the sections below run and measure it.

### Hierarchical parsing (`gather_clean_tex.py`, `hierarchy.py`)

- **Multi-file gathering.** The publication root is the `.tex` file with `\documentclass` (`standalone`/`subfiles` classes are ignored; ties are broken by `\begin{document}`, not being included by another file, a conventional name, then size). `\input`, `\include`, `\subfile` and `\import` are followed transitively in compilation order with TeX comment rules, so commented-out commands are plain text and files outside the compilation path are ignored. Conditionals such as `\iffalse` and `\newif` flags are evaluated, so draft text does not leak in. Missing includes are recorded as notices and mark the version `partial`; a version without TeX is recorded as `empty`.
- **Tree construction.** The root is the document node (holding the paper title). Headings get their level from the commands actually used (`\section`, `\subsection`, `\paragraph`, starred forms included), lists branch into items, and the leaves are sentences, block formulas and figures/tables. Acknowledgements and appendices are kept; the References section is excluded.
- **Sentence splitting.** SaT (`sat-3l-sm`, ONNX on CPU) splits prose. Mathematics, commands, abbreviations (`i.e.`, `et al.`, `Fig. 1`), URLs and decimals are masked before splitting, so a dot that does not end a sentence never creates a leaf. Predictions are cached per paragraph content, so paragraphs repeated across versions run the model once.

### Standardization and deduplication

- **LaTeX cleanup.** Comments are removed; author macros (`\newcommand`, `\def`, ...) are expanded; formatting-only commands (`\centering`, float placement such as `[htpb]`, `\midrule`, `\vspace`, ...) are dropped while the text inside decorative commands is kept; accents and quotes become Unicode; whitespace is normalized.
- **Math normalization.** All inline mathematics becomes `$...$`; all block mathematics (`\[ \]`, `$$`, `equation*`, `align`, `gather`, ...) becomes `\begin{equation}...\end{equation}`.
- **Reference extraction.** Entries come from the `.bib` files named by `\bibliography`/`\addbibresource` and from `\bibitem` blocks (rules for `\newblock`, quoted/italic titles and author-year styles). Titles, journals and authors are converted to Unicode; authors are written as `First Last`. Entries that still look mis-split can optionally be re-split with AnyStyle (section 4b).
- **Reference deduplication.** Entries of all versions of one paper are merged by DOI/arXiv ID, normalized title, citation key and fuzzy title (with a DOI veto). Fields are unioned instead of one entry being kept, one citation key is chosen per cluster, and `\cite` commands are rewritten to it.
- **Full-text deduplication.** After cleanup, elements with identical whitespace-normalized content share one `element_id` across versions; the `hierarchy` table keeps each version's own parent-child edges. Containers (headings, lists) are also matched by occurrence order, so two same-titled headings of one version never merge.

### Known limitations

- Conditionals that depend on macro arguments or run-time values (`\ifx#1\empty`, `\ifdim`) and macros built on TeX internals (`\@...`, `\expandafter`, `\csname`) are left unexpanded and reported as notices instead of being guessed.
- `\input{\macro}` (file name given by a macro) is not followed.
- Bibliography quality depends on the source style; entries whose structure cannot be recovered keep their raw text for the AnyStyle step.
- Statistics from a sample (`SAMPLE_N`) describe that sample only and must not be extrapolated to the whole corpus.


In [ ]:
# CONFIGURATION — edit before Run all
from pathlib import Path

RANGES = {
    1: ("2609", 27834, 32833),
    2: ("2609", 17834, 22833),
    3: ("2608", 14007, 19006),
    4: ("2608", 4007, 9006),
    5: ("2511", 6709, 11708),
}
RUN_RANGES = "local"      # [3], [1, 2], "all" or "local" (find papers from directories/state)
SAMPLE_N = None             # first N IDs of each range; None = all; local: cap on the total number of papers
WORKER = None             # None: named after the ranges; use different names when running concurrently

REPO_DIR = None           # local: find the repo from the current directory; or an absolute path
REPO_URL = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH = "main"           # branch that Colab clones; it must contain the parser modules
INSTALL_DEPENDENCIES = None  # None: install on Colab, only check locally
MOUNT_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intro2DS-Group3"
DATA_DIR_OVERRIDE = None    # e.g. r"C:/Intro2DS-Group3-Source/data"
DATA_TAR = None              # None: on Colab, use data.tar.gz/.tgz/.tar in DRIVE_PROJECT_DIR if present;
                             # a specific path (relative to DRIVE_PROJECT_DIR); False = always read the data directory
LOCAL_DATA_DIR = "/content/parser_data"  # where the tar is extracted; local disk, lost when the runtime resets
OUTPUT_DIR_OVERRIDE = None   # default statistics_data/parser, separate from the scraper output
SOURCE_STATE_PATHS = None    # None: find state automatically; with ranges, unrelated range workers are skipped

RUN_SMOKE_TEST = True
RUN_PIPELINE = True
RESUME = True               # reuse valid ok/empty/partial checkpoints
RETRY_PARTIAL = False        # True: parse partial papers again even if their sources did not change
CHECKPOINT_VALIDATION = "fast"  # "fast": size/mtime, hash on change; "full": always hash the content
DEBUG_REPORTS = False        # True: add detailed reports, RAM/timing history and combined warnings
ENABLE_ENRICHMENT = False   # offline by default; True allows CrossRef and LLM if a key is set
FUZZY_THRESHOLD = 0.92
REFINE_BIBTEX = True        # post-process bibtex after parsing (section 4b); cached, so reruns are fast
REFINE_WITH_ANYSTYLE = True # re-split suspicious \bibitem entries; Colab installs Ruby + anystyle-cli (a few minutes)
REFINE_CACHE_DIR = None     # None: <OUTPUT_DIR>/refine_cache; keep it on Drive to reuse across sessions
RUBY_BIN_DIR = None         # local only: Ruby's bin folder if it is not on PATH, e.g. r"D:\Tools\Ruby33\bin"
SENTENCE_CACHE_MB = 4       # MiB of cached sentence predictions per paper; 0 disables it; excludes model RAM
NUM_WORKERS = "auto"        # chosen from CPU/RAM; 1 = sequential; an integer = desired limit
MAX_WORKERS = None          # extra cap on the number of processes; None = CPU limit
ADAPTIVE_RAM = True         # adjust the workload and reclaim idle workers when RAM runs low
RESERVE_RAM_GB = None       # None: reserve RAM for the system automatically; or GiB to reserve
RAM_PER_WORKER_GB = 1.5     # minimum estimate per worker, raised from the measured RSS
RAM_WAIT_TIMEOUT_SEC = 60   # when the RAM wait times out, finished work is saved and the rest runs later
PAPER_TIMEOUT_SEC = 900     # a paper running longer is stopped and recorded as an error; None = no limit (parallel only)
MAX_RUN_HOURS = 11.0        # checked between papers; None = no limit
MONITOR_EVERY_SEC = 1.0
DISK_EVERY_PAPERS = None    # only used when DEBUG_REPORTS=True; None: scan at the end of the session


## 1. Running locally and on Google Colab

**Local:** use Python 3.12 or later, run `pip install -r src/requirements.txt`, and open the notebook from the repository or set `REPO_DIR`. Data is read from `<repo>/data` by default and results are written to `<repo>/statistics_data/parser`. To process another directory, set `DATA_DIR_OVERRIDE` and `RUN_RANGES = "local"`. On a machine with little free RAM, set `NUM_WORKERS = 1`: the parallel scheduler refuses to start a worker it cannot fit in memory.

**Colab:** choose a CPU runtime. The notebook clones the repository if needed, installs the parser libraries at the versions in `src/requirements.txt`, then mounts Drive. Push the parser code to the `BRANCH` branch before opening it on Colab. If the repository already exists, the notebook uses that code and never pulls over local changes; if that clone is on a branch other than `BRANCH`, the notebook stops and asks you to delete the repository folder or check out the right branch, so code from another branch is never run by mistake. By default it reads `MyDrive/Intro2DS-Group3/data` and writes `MyDrive/Intro2DS-Group3/statistics_data/parser`.

**Compressed data source (optional):** if `DRIVE_PROJECT_DIR` contains `data.tar.gz`, `data.tgz` or `data.tar`, the notebook extracts it to `LOCAL_DATA_DIR` on the runtime disk and reads the data from there; otherwise it reads the `data` directory on Drive directly. The extracted copy is reused within the same runtime while the archive is unchanged. Output is always written to Drive so checkpoints survive a runtime reset. The archive is a snapshot of `data` at creation time, so recreate it when the source data changes. Checkpoint signatures do not depend on the location of the data directory, so switching between the two sources keeps the checkpoints.

**After updating the code:** restart the kernel/runtime, then Run all to load the new modules. Old results do not need to be deleted: changes to the processing code, schema or libraries automatically invalidate incompatible checkpoints, while changes to reporting/scheduling alone do not force a re-parse. Run all in a kernel that still holds the old modules does not reload the Python code.

**Resuming:** each finished paper has its own Parquet partition and checkpoint. By default `CHECKPOINT_VALIDATION="fast"` compares file size and modification time and hashes again only when a file changed or an old checkpoint needs upgrading. `"full"` always hashes the content, which also catches changes that keep the size and modification time. Valid `ok/empty/partial` checkpoints are reused; `partial` keeps its status and warnings and is not counted as a full success. Set `RETRY_PARTIAL=True` to retry them; `error` is always retried. Changes to the sources, configuration or processing code make a paper be checked/processed again. Checkpoints from a compatible older parser are fully hashed once to migrate their format; incompatible checkpoints are parsed again. Do not run two notebooks with the same `WORKER` into one output directory. The run can be stopped with Interrupt; the paper in progress is run again in the next session. The time limit is only checked between papers.

**Parallelism:** the default is `NUM_WORKERS="auto"`. Each process handles a whole paper and reuses its own SaT model. Within a paper, predictions for passages with the same SaT input are cached within the `SENTENCE_CACHE_MB` budget, then cleared before the bibliography is unified. Warnings and errors are never cached. The scheduler runs one paper to measure RAM before adding workers, within the CPU and available RAM limits. Set `MAX_WORKERS` for an extra cap, or `NUM_WORKERS=1` to go back to sequential mode. On Windows/Colab, workers are spawned from the Python modules and do not rerun notebook cells.

**Network:** `ENABLE_ENRICHMENT=False` disables enrichment; the first SaT weight download and library installation still need the network. To run fully offline, install the libraries and cache the weights beforehand. A Gemini API key, if used, is read from an environment variable or loaded into the environment from Colab Secrets; the notebook never writes the key to the configuration or logs.

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import PureWindowsPath


def resolve_config_path(value):
    """Accept native paths and mounted Windows drive paths in a WSL kernel."""
    text = str(value)
    windows = PureWindowsPath(text)
    if sys.platform != "win32" and windows.drive:
        if len(windows.drive) != 2 or windows.drive[1] != ":" or not windows.root:
            raise ValueError(f"A Windows path needs an absolute drive, e.g. C:/folder: {text}")
        mount = Path("/mnt") / windows.drive[0].lower()
        if not mount.is_dir():
            raise ValueError(
                f"Kernel {sys.executable} ({sys.platform}) cannot access the Windows path {text}. "
                "On Colab use a /content or Drive path; on WSL use a drive mounted at /mnt/<drive>.")
        return mount.joinpath(*windows.parts[1:]).resolve()
    return Path(text).expanduser().resolve()

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if REPO_DIR is not None:
    REPO_ROOT = resolve_config_path(REPO_DIR)
elif IN_COLAB:
    REPO_ROOT = Path("/content/Intro2DS-Group3-Source")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
    else:
        current_branch = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
                                        capture_output=True, text=True).stdout.strip()
        if current_branch != BRANCH:
            raise RuntimeError(f"{REPO_ROOT} is on branch {current_branch!r}, not BRANCH={BRANCH!r}. "
                               f"Run !rm -rf {REPO_ROOT} (or git checkout {BRANCH}), restart the runtime, then Run all.")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates if (p / "src" / "hierarchy.py").is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("Repository not found. Set REPO_DIR in the configuration cell.")

# Accept the src directory itself, while keeping REPO_ROOT at the project root.
if not (REPO_ROOT / "src" / "hierarchy.py").is_file() and (REPO_ROOT / "hierarchy.py").is_file():
    REPO_ROOT = REPO_ROOT.parent
print("Kernel:", sys.executable, "| Platform:", sys.platform, "| Repo:", REPO_ROOT)

required_files = ["parser_pipeline.py", "parser_reporting.py", "parser_parallel.py", "gather_clean_tex.py", "bibtex.py", "hierarchy.py", "parser_storage.py"]
missing_files = [name for name in required_files if not (REPO_ROOT / "src" / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Modules {missing_files} not found in {REPO_ROOT / 'src'}. "
        f"Kernel: {sys.executable} ({sys.platform}). "
        "REPO_DIR must point to the repository root that contains src; rerun the CONFIGURATION cell before this one. "
        "If the path is correct, update all parser modules.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

parser_packages = {"pandas", "pyarrow", "matplotlib", "bibtexparser", "psutil", "wtpsplit", "onnxruntime"}
requirements = [line.strip() for line in (REPO_ROOT / "src" / "requirements.txt").read_text().splitlines()
                if line.strip().split("==")[0] in parser_packages]
if {r.split("==")[0] for r in requirements} != parser_packages:
    raise ValueError("requirements.txt does not define all parser libraries.")
install = IN_COLAB if INSTALL_DEPENDENCIES is None else INSTALL_DEPENDENCIES
versions_before = {name: getattr(sys.modules.get(name), "__version__", None) for name in parser_packages}
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", *requirements], check=True)

dependency_rows = []
for requirement in requirements:
    name, expected = requirement.split("==")
    try:
        actual = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(f"{name} is missing. Run pip install -r src/requirements.txt, then restart the kernel.")
    dependency_rows.append({"package": name, "required": expected, "installed": actual})
    if versions_before[name] is not None and versions_before[name] != actual:
        raise RuntimeError(f"{name} was imported before the update. Restart the kernel/runtime, then Run all again.")
    if actual != expected:
        print(f"Note: {name}={actual}, differs from the reproducible version {expected}.")
print("Environment:", "Google Colab CPU" if IN_COLAB else "Local", "| Repo:", REPO_ROOT)

In [ ]:
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

project_root = Path(DRIVE_PROJECT_DIR) if IN_COLAB and MOUNT_DRIVE else REPO_ROOT

import shutil
import time

# Stored in run_parser's run information so end-to-end time includes data preparation.
DATA_PREPARATION = {"data_source": "directory"}


def extracted_data_dir():
    """Extract the data tar to the local disk; return None when no tar is used."""
    if DATA_DIR_OVERRIDE or DATA_TAR is False:
        return None
    if DATA_TAR is None:
        if not (IN_COLAB and MOUNT_DRIVE):
            return None
        found = [project_root / name for name in ("data.tar.gz", "data.tgz", "data.tar") if (project_root / name).is_file()]
        if not found:
            return None
        archive = found[0]
    else:
        archive = Path(DATA_TAR).expanduser()
        archive = archive if archive.is_absolute() else project_root / archive
        if not archive.is_file():
            print(f"{archive} not found; reading the data directory as usual.")
            return None
    target = Path(LOCAL_DATA_DIR)
    info = archive.stat()
    stamp = f"{archive}|{info.st_size}|{info.st_mtime_ns}"
    marker = target.with_name(target.name + ".source")
    seconds_file = target.with_name(target.name + ".seconds")
    DATA_PREPARATION.update(data_source="tar", archive=archive.name, archive_bytes=info.st_size)
    if target.is_dir() and marker.is_file() and marker.read_text(encoding="utf-8") == stamp:
        print(f"Reusing data extracted from {archive.name} at {target}")
        first = seconds_file.read_text(encoding="utf-8").strip() if seconds_file.is_file() else ""
        DATA_PREPARATION.update(extract_sec_this_session=0.0, reused_extraction=True,
                                extract_sec_first=float(first) if first else None)
    else:
        shutil.rmtree(target, ignore_errors=True)
        marker.unlink(missing_ok=True)
        target.mkdir(parents=True)
        if shutil.disk_usage(target).free < info.st_size:
            print(f"Not enough local disk space for {archive.name}; reading the data directory as usual.")
            DATA_PREPARATION.clear(); DATA_PREPARATION["data_source"] = "directory"
            return None
        print(f"Extracting {archive} ({info.st_size / 2**30:.2f} GiB) → {target} ...")
        started = time.perf_counter()
        try:
            subprocess.run(["tar", "-xf", str(archive), "-C", str(target)], check=True)
        except (OSError, subprocess.CalledProcessError) as exc:
            shutil.rmtree(target, ignore_errors=True)
            print(f"Extraction failed ({exc}); reading the data directory as usual.")
            DATA_PREPARATION.clear(); DATA_PREPARATION["data_source"] = "directory"
            return None
        elapsed = time.perf_counter() - started
        marker.write_text(stamp, encoding="utf-8")
        seconds_file.write_text(f"{elapsed:.3f}", encoding="utf-8")
        DATA_PREPARATION.update(extract_sec_this_session=round(elapsed, 3), reused_extraction=False,
                                extract_sec_first=round(elapsed, 3))
        print(f"Extraction finished after {elapsed:.0f}s")
    # The tar may hold a data/ directory at its root or the paper directories directly.
    return target / "data" if (target / "data").is_dir() else target


DATA_DIR = (resolve_config_path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE
            else extracted_data_dir() or project_root / "data")
OUTPUT_DIR = resolve_config_path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else project_root / "statistics_data" / "parser"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory {DATA_DIR} does not exist. Check Drive or DATA_DIR_OVERRIDE.")
if OUTPUT_DIR.resolve() == DATA_DIR.resolve():
    raise ValueError("OUTPUT_DIR must differ from DATA_DIR to keep parser output apart from the scraper sources.")

import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from src.parser_pipeline import discover_publications, parse_publication
from src.parser_reporting import run_parser

display(pd.DataFrame(dependency_rows))
print("Input:", DATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. Scope selection and input check

Ranges keep the definitions of `scraper.ipynb`. A sample only changes the reported scope and must not be extrapolated to all 25,000 IDs. `local` mode finds papers through existing directories and state, including empty versions that only exist in the state; range mode keeps IDs without data so the coverage denominator is not reduced.

The sample limit is applied before any paper's metadata is read. With target IDs, the notebook checks each paper directory directly instead of listing the whole store. When ranges are selected, the state of the matching range workers (including combined-range workers) is found automatically; shared or custom-named state is still read. An explicit `SOURCE_STATE_PATHS` overrides this filter; use it if data was moved between workers or the naming convention changed.

The version list comes from the metadata, the Milestone 1 state and the actual directories. Versions confirmed as `no_tex` are recorded as `empty`; a missing directory without evidence is never assumed to be empty. IDs without any version information use a v1 record to represent the missing input, which is not evidence that the paper has one version. Metadata errors, missing sources and ambiguous bibliographies are kept in the check results and statuses.

In [ ]:
import time
discovery_started = time.perf_counter()

if SAMPLE_N is not None and (type(SAMPLE_N) is not int or SAMPLE_N <= 0):
    raise ValueError("SAMPLE_N must be a positive integer or None.")
if RUN_RANGES == "local":
    selected_ranges = []
    jobs = discover_publications(DATA_DIR, source_state_paths=SOURCE_STATE_PATHS, limit=SAMPLE_N)
else:
    selected_ranges = sorted(RANGES) if RUN_RANGES == "all" else list(RUN_RANGES)
    if not selected_ranges or len(set(selected_ranges)) != len(selected_ranges) or any(r not in RANGES for r in selected_ranges):
        raise ValueError("RUN_RANGES must be a list of valid ranges, 'all' or 'local'.")
    targets = []
    for r in selected_ranges:
        month, first, last = RANGES[r]
        end = last + 1 if SAMPLE_N is None else min(last + 1, first + SAMPLE_N)
        targets.extend(f"{month}.{number:05d}" for number in range(first, end))
    if len(targets) != len(set(targets)):
        raise ValueError("The range definitions overlap.")
    jobs = discover_publications(DATA_DIR, target_ids=targets, source_state_paths=SOURCE_STATE_PATHS,
                                 state_ranges=[f"R{r}" for r in selected_ranges])

def range_label(arxiv_id):
    for r, (month, first, last) in RANGES.items():
        if arxiv_id.startswith(month + ".") and arxiv_id.split(".")[-1].isdigit() and first <= int(arxiv_id.split(".")[-1]) <= last:
            return f"R{r}"
    return "local"

for job in jobs:
    job["range"] = range_label(job["arxiv_id"])
WORKER_NAME = WORKER or ("local" if RUN_RANGES == "local" else "+".join(f"R{r}" for r in selected_ranges))
print(f"Input check: {time.perf_counter() - discovery_started:.1f} seconds")
print(f"Worker: {WORKER_NAME} | {len(jobs):,} IDs | {sum(len(j['version_dirs']) for j in jobs):,} expected versions")
preview = [{"arxiv_id": j["arxiv_id"], "range": j["range"], "versions": len(j["version_dirs"]),
            "directory": str(j.get("paper_dir") or "none"), "detected errors": "; ".join(j.get("discovery_errors", []))}
           for j in jobs[:20]]
display(pd.DataFrame(preview))

## 3. Smoke test before the corpus run

An offline fixture has two versions with the same sentence but different headings/citation keys, plus one version without TeX. It checks that the bibliography keeps both the year and the DOI and that the shared sentence is attached to the right parent. The sentence-splitting model is mocked to test the integration; SaT accuracy is not evaluated here.

In [ ]:
if RUN_SMOKE_TEST:
    from tempfile import TemporaryDirectory
    from unittest.mock import patch
    with TemporaryDirectory() as temporary:
        base = Path(temporary)
        paper = base / "2301-00001"
        paper.mkdir()
        (paper / "metadata.json").write_text(json.dumps({"arxiv_id": "2301.00001", "num_versions": 3}), encoding="utf-8")
        for version, title, key, fields in [(1, "Introduction", "old", "year={2017}"),
                                           (2, "Background", "new", "doi={10.1000/attention}")]:
            folder = paper / f"v{version}"
            folder.mkdir()
            (folder / "main.tex").write_text(r"\documentclass{article}\begin{document}" +
                rf"\section{{{title}}}We study attention \cite{{{key}}}.\bibliography{{refs}}" +
                r"\end{document}", encoding="utf-8")
            (folder / "refs.bib").write_text(f"@article{{{key},title={{Attention Is All You Need}},"
                f"author={{Vaswani, Ashish}},{fields}}}", encoding="utf-8")
        (paper / "v3").mkdir()
        with patch("src.hierarchy._sentence_model") as model:
            model.return_value.split.side_effect = lambda text, **kwargs: [text]
            demo = parse_publication(discover_publications(base)[0], enrich=False)
        assert len(demo["bibtex"]) == 1 and demo["bibtex"][0]["year"] == 2017
        assert demo["bibtex"][0]["doi"] == "10.1000/attention"
        sentence = [r for r in demo["elements"] if r["kind"] == "sentence"]
        assert len(sentence) == 1
        edges = [r for r in demo["hierarchy"] if r["element_id"] == sentence[0]["element_id"]]
        assert len(edges) == 2 and edges[0]["parent_id"] != edges[1]["parent_id"]
        assert any(r["version"] == 3 and r["status"] == "empty" for r in demo["version_status"])
        print("Integration test passed: references unified, sentence shared, correct parent per version, empty version kept.")

## 4. Running the pipeline with checkpoints

Papers can be parsed in parallel by several processes; all versions of a paper are still processed and unified in the same worker, so the parser method is unchanged. Workers reuse a CPU SaT model and write Parquet in batches to their own partitions, while the main process checks the metadata and writes checkpoints and statistics. Large results never pass through the pipe; only stats, statuses and file metadata do. Checkpoint checking has its own progress output before parsing starts. By default only the three dataset tables are measured; checkpoint/log directories are not scanned. `DEBUG_REPORTS=True` enables RAM/timing history, extra statistics and output directory scans. If the partitions and the scope are unchanged, fast mode reuses the three combined tables instead of reading every partition again. The file inventory from the source check is reused for TeX gathering. Do not modify the sources while the parser is running. The number of papers in flight is bounded, so the whole corpus is never queued at once. Combined tables are assembled in batches and keep a stable paper order even when work finishes out of order.

With `ADAPTIVE_RAM=True`, the scheduler reserves system RAM, measures worker RSS and updates its estimate. When RAM drops, it stops handing out papers and reclaims idle processes; papers in progress are completed. If no paper is running and RAM is still insufficient after `RAM_WAIT_TIMEOUT_SEC`, the session saves the finished work and reports an error so it can be resumed later. This lowers the risk of running out of RAM but is not a hard RAM cap: a very large paper can still exceed the estimate. `NUM_WORKERS=1` runs sequentially and does not use the RAM scheduler.

`PAPER_TIMEOUT_SEC` sets the maximum time for one paper, counted from when it is handed out (including starting a new worker and the first model load). A paper over the limit is stopped, recorded as `error` with the reason, and retried in a later session; a replacement worker is created when needed. The limit applies only to parallel runs, because `NUM_WORKERS=1` runs inside the notebook process.

Library deprecation warnings (`DeprecationWarning`, `FutureWarning`, ...) do not reflect the data, so they are only counted in each paper's `dependency_warning_count`. One content repeated under several parents in the same version is reported as one warning per version and does not set `review_required`, because every edge is kept. Paths in messages and checkpoint signatures are relative to the data directory, so switching between the `data` directory on Drive and the extracted copy keeps the checkpoints.

When `MAX_RUN_HOURS` is reached, no new paper is handed out, but papers already handed out are completed. Interrupt cleans up the processes created by the session; papers without a checkpoint are marked pending. The history of worker scaling and RAM waits is in `events.jsonl` and in the session's parallel information.

`paper_success` is true only for `ok`: every version with TeX sources was parsed without losing content. A paper submitted as PDF only (`empty`) and a version arXiv reports as not found are coverage, not success: `success_rate_with_tex` leaves PDF-only papers out, and `papers_ok_bibliography_not_in_corpus` counts `ok` papers whose bibliography is absent from the `.tex`/`.bib` corpus (for example only a `.bbl` was submitted). Like BibTeX and LaTeX, the parser keeps an undefined `@string` as its literal text, skips only a malformed `.bib` entry, resolves a key defined twice as the compiled document does, and treats an included TikZ/PGF/style file (not kept by the corpus) and an undecidable conditional in the preamble or a macro definition as notices. `ok` may come with notices about merged metadata or repeated relations that were kept; notices do not cause a re-parse. `review_required=True` means the metadata/schema quality still needs review, independently of the pipeline having completed.

`partial` is for results that are incomplete or uncertain: a missing `.tex` content file, an undecidable conditional in the document body, broken LaTeX, skipped malformed `.bib` entries or an unresolvable citation key. `error` means a processing step failed. `partial` papers with a valid checkpoint are reused by default to avoid repeating work that does not improve the result; they are still not counted as full successes. Set `RETRY_PARTIAL=True` to retry them. `error` papers are always retried. Filter `per_version.csv` by status to find what needs fixing. Detailed diagnostics remain in `tables/<WORKER>/papers/<ID>/warnings.jsonl`; they are combined into `warnings.csv` and `issues.csv` only when `DEBUG_REPORTS=True`. A reference list cannot be reconstructed from a missing `.bbl` file by relabeling its status.

In [ ]:
def show_checkpoint_progress(done, total, reused):
    if done == 1 or done % 100 == 0 or done == total:
        print(f"[Checkpoint {done:,}/{total:,}] Reusable: {reused:,}", flush=True)

def show_progress(done, total, stats):
    if done == 1 or done % 25 == 0 or done == total:
        print(f"[{done:,}/{total:,}] {stats['arxiv_id']} — {stats['status']} | "
              f"element: {stats.get('elements_before', 0)} → {stats.get('elements_after', 0)}")
        if stats['status'] not in {'ok', 'empty'}:
            print("  Reason:", stats.get('message', '')[:700])

if RUN_PIPELINE:
    result = run_parser(jobs, OUTPUT_DIR, worker=WORKER_NAME, ranges=RANGES,
        enrich=ENABLE_ENRICHMENT, fuzzy_threshold=FUZZY_THRESHOLD, resume=RESUME,
        max_run_hours=MAX_RUN_HOURS, monitor_interval=MONITOR_EVERY_SEC,
        disk_every=DISK_EVERY_PAPERS, progress=show_progress, debug_reports=DEBUG_REPORTS,
        retry_partial=RETRY_PARTIAL, checkpoint_validation=CHECKPOINT_VALIDATION,
        checkpoint_progress=show_checkpoint_progress, sentence_cache_mb=SENTENCE_CACHE_MB,
        num_workers=NUM_WORKERS, max_workers=MAX_WORKERS, adaptive_ram=ADAPTIVE_RAM,
        reserve_ram_gb=RESERVE_RAM_GB, ram_per_worker_gb=RAM_PER_WORKER_GB,
        ram_wait_timeout_sec=RAM_WAIT_TIMEOUT_SEC, paper_timeout_sec=PAPER_TIMEOUT_SEC,
        data_preparation=DATA_PREPARATION)
    summary = result["summary"]
else:
    summary_path = OUTPUT_DIR / "report_exports" / WORKER_NAME / "summary_stats.json"
    if not summary_path.exists():
        raise FileNotFoundError("No reports available. Set RUN_PIPELINE=True for the initial run.")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))

REPORTS_DIR = OUTPUT_DIR / "report_exports" / WORKER_NAME
TABLES_DIR = OUTPUT_DIR / "tables" / WORKER_NAME
display(pd.DataFrame([{
    "IDs in scope": summary["target_ids"], "successful papers": summary["papers_success"],
    "checkpoints reused": summary["run"]["papers_reused"],
    "success rate": summary["success_rate"], "success rate (papers with TeX)": summary.get("success_rate_with_tex"),
    "PDF-only papers": summary.get("papers_empty"), "ok papers with bibliography": summary.get("papers_ok_with_bibliography"),
    "elements after deduplication": summary["elements_after"],
    "references after deduplication": summary["bibtex_after"], "stop reason": summary["run"]["stop_reason"],
    "run duration (seconds)": summary["run"]["duration_sec"],
    "average parse time per paper (seconds)": summary.get("average_parse_sec"),
    "data extraction this session (seconds)": summary["run"].get("data_preparation", {}).get("extract_sec_this_session"),
    "peak RSS (MiB)": summary["run"]["peak_rss_bytes"] / 2**20,
    "Parquet tables (MiB)": summary["table_bytes"] / 2**20,
}]))
if DEBUG_REPORTS and summary["run"].get("parallel"):
    print("CPU/RAM scheduling:", summary["run"]["parallel"])
if summary["run"]["stop_reason"] == "error":
    print("Run error:", summary["run"].get("error"), "— check parser.log before using the tables.")

## 4b. Post-processing `bibtex` with AnyStyle

This optional step runs **after** parsing, over all papers of the worker, in batches; it does not modify the parser's partitions or checkpoints. AnyStyle (a citation parser based on a CRF model) only receives `\bibitem` entries that show signs of a wrong split: a missing title or authors, a title that looks like a journal/conference name, or unusually long author names. A title returned by AnyStyle is accepted only if it occurs in the original `\bibitem` text. All entries are sent in one CLI call in chunks of 2,000 lines, because starting Ruby is the main cost.

The only input is the paper's own source text. Neither the parser nor this step uses the crawled reference list (`references.json`) or external lookups: Milestone 3 matches these `bibtex` entries against that list, so any field derived from it would leak the labels.

Results are cached in `REFINE_CACHE_DIR/anystyle.jsonl`; later runs call the CLI only for missing entries. `tables/<WORKER>/bibtex.parquet` is replaced by the refined table (marked in the Parquet metadata), the parser's table is kept as `bibtex_parsed.parquet`, and every change (field, old/new value, reason) is in `bibtex_refinement.jsonl`. Running `run_parser` again rewrites `bibtex.parquet` from the partitions, so this section must be run again as well. This step does not deduplicate again. Its time is reported separately and is not included in the parse time.

In [ ]:
if REFINE_BIBTEX:
    if RUBY_BIN_DIR:
        # Only for this kernel session: nothing is changed in the Windows PATH.
        os.environ["PATH"] = str(Path(RUBY_BIN_DIR)) + os.pathsep + os.environ["PATH"]
    if REFINE_WITH_ANYSTYLE and shutil.which("anystyle") is None:
        if IN_COLAB:
            tick = time.perf_counter()
            installed = subprocess.run("apt-get -qq update && apt-get -qq install -y ruby-dev build-essential "
                                       "> /dev/null && gem install --no-document anystyle-cli > /dev/null",
                                       shell=True, capture_output=True, text=True)
            # A failed install does not stop Run all: the AnyStyle step skips itself and records the reason.
            print(f"Installing anystyle-cli: {'done' if installed.returncode == 0 else 'failed'} "
                  f"after {time.perf_counter() - tick:.0f} s", installed.stderr.strip()[-300:])
        else:
            print("No anystyle command (needs Ruby and `gem install anystyle-cli`); skipping the AnyStyle step.")
    from src.bibtex_refine import refine_bibliography
    tick = time.perf_counter()
    refinement = refine_bibliography(
        OUTPUT_DIR, worker=WORKER_NAME,
        cache_dir=REFINE_CACHE_DIR or OUTPUT_DIR / "refine_cache",
        use_anystyle=REFINE_WITH_ANYSTYLE, progress=print)
    print(f"bibtex post-processing: {time.perf_counter() - tick:.1f} s")
    if refinement["anystyle"].get("skipped"):
        print("AnyStyle:", refinement["anystyle"]["skipped"])
    display(pd.DataFrame([{k: v for k, v in refinement["anystyle"].items() if k != "skipped"}]))
    display(pd.Series(refinement["changes_by_field"], name="changes", dtype="int64"))
    print(f"{refinement['rows_changed']:,}/{refinement['rows']:,} entries refined; "
          f"details: {OUTPUT_DIR / 'tables' / WORKER_NAME / 'bibtex_refinement.jsonl'}")


## 5. Statistics required by the milestone

By default the notebook exports `per_paper.csv`, `per_version.csv` and `summary_stats.json`: success rate, version coverage (including `empty`), element/reference counts before and after deduplication, wall time, average parse time per paper, maximum RAM and final dataset size.

For a paper reused from a checkpoint, `elapsed_sec` is the saved parse time. `average_parse_sec` averages the saved/measured parse times; it differs from the time of the current resume. `run.duration_sec` measures the current session, including checkpoint checks, parsing, table assembly and the main report export. For the final benchmark on **Colab CPU**, use `RESUME=False` to measure one full parse; do not add up parallel worker times as wall time.

RAM is the RSS of the Python process and its children, sampled to keep the maximum in memory. Normal mode does not write RAM history to Drive. `table_bytes` counts only the three dataset tables; checkpoints, logs and partitions are not added to the dataset size again.

Only `DEBUG_REPORTS=True` exports category/range statistics, combined warnings, per-stage timings, RAM/disk history and the extra figures. Old debug files left in the directory are not read in normal mode.


In [ ]:
per_paper = pd.read_csv(REPORTS_DIR / "per_paper.csv", dtype={"arxiv_id": str})
per_version = pd.read_csv(REPORTS_DIR / "per_version.csv", dtype={"arxiv_id": str})

display(per_paper.head(20))
display(per_version.head(20))
issues = per_version[~per_version["status"].isin(["ok", "empty"])]
print(f"{len(issues):,} versions are partial, missing, failed, or pending.")
display(issues.head(20))

if DEBUG_REPORTS:
    display(pd.read_csv(REPORTS_DIR / "range_stats.csv"))
    print("Detailed quality notices and warnings:")
    display(pd.read_csv(REPORTS_DIR / "warnings.csv", dtype={"arxiv_id": str}).head(30))


In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
per_paper["status"].value_counts().plot.bar(ax=axes[0], title="Paper status") if len(per_paper) else axes[0].set_title("No papers")
axes[0].set_ylabel("Number of papers")
if len(per_version):
    per_version["status"].value_counts().plot.bar(ax=axes[1], title="Version status")
else:
    axes[1].set_title("No versions")
axes[1].set_ylabel("Number of versions")
pd.DataFrame({"Before": [summary["elements_before"], summary["bibtex_before"]],
              "After": [summary["elements_after"], summary["bibtex_after"]]},
             index=["Element", "Reference"]).plot.bar(ax=axes[2], title="Before and after deduplication")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "coverage_dedup.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

if DEBUG_REPORTS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    current = per_paper[per_paper["reused"].eq(False)]
    times = pd.to_numeric(current["elapsed_sec"], errors="coerce").dropna()
    if len(times):
        axes[0].hist(times, bins=min(30, max(1, len(times))))
    axes[0].set(title="Parsing time for papers without reused checkpoints", xlabel="Seconds", ylabel="Number of papers")
    monitor_path = REPORTS_DIR / "monitor.csv"
    if monitor_path.exists():
        monitor = pd.read_csv(monitor_path)
        for run_id, group in monitor.groupby("run_id"):
            axes[1].plot(pd.to_datetime(group["minute"], utc=True), group["max_rss_bytes"] / 2**20,
                         label=run_id)
        if len(monitor):
            axes[1].legend(fontsize=7)
    axes[1].set(title="Peak RSS per minute", xlabel="UTC time", ylabel="MiB")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "performance.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 6. Output tables and how to read them

```text
statistics_data/parser/
├── tables/<WORKER>/
│   ├── elements.parquet       # element_id, arxiv_id, kind, content
│   ├── hierarchy.parquet      # arxiv_id, version, element_id, parent_id
│   ├── bibtex.parquet         # BibTeX fields as in docs/requirement/parsing.md (refined if section 4b ran)
│   ├── bibtex_parsed.parquet  # table produced by the parser, before section 4b
│   ├── bibtex_refinement.jsonl # every change made by section 4b
│   └── papers/<ID>/           # partition, key_map.json, bibtex_debug.jsonl, warnings.jsonl
├── report_exports/<WORKER>/
│   ├── per_paper.csv          # success, coverage, counts before/after, parse time
│   ├── per_version.csv        # status of each version, including empty/partial/error/pending
│   ├── summary_stats.json     # totals, wall time, average time, peak RAM, dataset size
│   ├── README.md
│   └── figures/               # figures produced by the report cells
├── refine_cache/              # anystyle.jsonl (section 4b)
├── run_logs/<WORKER>/         # runs.jsonl: start/end of each session; parser.log: errors
└── .state/<WORKER>/           # checkpoints for resuming
```

`kind` and `entry_type` are dictionary-encoded, `version`/`year` are integers, and `authors` is a list of strings. Empty tables keep their schema. The root has a row in `elements` but no parent edge. Do not read all of `papers/` together with the combined tables in one Parquet glob, as that counts rows twice.

Per-paper checkpoints and diagnostics are kept for resuming and debugging, but they do not need to be read or combined just to export the default report. Detailed CSV/debug logs are only created or read with `DEBUG_REPORTS=True`. The summary report describes exactly the current export scope; the CSV files can serve as input for tables/figures generated while building the Jupyter Book.

**`elements` content.** The `document` root of each version holds the paper title (from `\title`, without `\thanks` and line breaks); without a title its content is empty. The `\begin`/`\end` tags of environments outside the math, figure, list and verbatim groups (e.g. custom theorem environments) are block boundaries and never appear in sentence content. Text before and after a display formula forms separate sentences, because the formula is a leaf of its own.

**`bibtex` content.** `title`, `journal` and `authors` are converted from LaTeX to Unicode (accents such as `{\"u}` become `ü`; case-protecting braces and formatting commands are removed); `$...$` math is kept. Author names are written as `First Last`; organization names braced in the BibTeX source are kept as is. The arXiv ID is taken from `eprint`, a `10.48550/arXiv.*` DOI, an arxiv.org URL or an `arXiv:...` string in the text fields, and is used as an identity key during unification. The year is read from `year` or `date`; if no version has a year but an arXiv ID exists, the year is inferred from the ID (first submission year) and recorded as `inferred_fields=["year"]` in `bibtex_debug.jsonl`. The table keeps every bibliography entry; the `cited` flag in `bibtex_debug.jsonl` and `bibtex_cited` in the statistics show which entries are `\cite`d in the parsed content (`\nocite{*}` counts as all).

**Splitting `\bibitem`.** Entries with `\newblock` are split by BibTeX blocks: the first block is the authors, the second the title, the rest the venue. Entries without `\newblock` use quotation marks, italics and punctuation (including the form `Author. Title. In \emph{Conference}`); the author–year style (`Author. 2024. Title.`) is recognized so the year does not end up in the last author's name. Only the source text is used; the crawled reference list is not involved because it is the data Milestone 3 matches against. `bibtex_debug.jsonl` keeps the `\bibitem` text (`raw_text`) for section 4b. Per-paper statistics include `cite_keys` (number of `\cite`d keys) and `cite_keys_unresolved` (keys without any entry, usually because the bibliography only exists as a `.bbl` that Milestone 1 does not keep); `summary_stats.json` sums them under `bibliography`.

**Elements shared across versions.** Leaves (sentences, formulas, figures/tables) with the same content after cleanup share one `element_id`. Nodes with children (root, heading, list, item) are additionally matched by their order of appearance in the version: the k-th "Proof" heading of v1 and of v2 share an ID, but two "Proof" headings or two lists of the same version are separate nodes, so their children are never merged.

In [ ]:
# Read only the first batch as a sample, to avoid loading the whole corpus into RAM.
for name in ("elements", "hierarchy", "bibtex"):
    path = TABLES_DIR / f"{name}.parquet"
    with pq.ParquetFile(path) as parquet:
        print(f"\n{name}: {parquet.metadata.num_rows:,} rows | {path}")
        print(parquet.schema_arrow)
        first_batch = next(parquet.iter_batches(batch_size=5), None)
        if first_batch is not None:
            display(first_batch.to_pandas())
print("Report exports:", REPORTS_DIR)